# 📓 Update dataset with last boxscores and matches

# Please first run full pipeline until merge_clean_seasons_boxscores to have base data to work with and merge

In [1]:

import os
import pandas as pd
from datetime import datetime
from src.config import *
from src.utils import *
from src.nba_scrapping import *



In [2]:
# ⚙️ Initialisation du run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_season = '2025-26'  # À rendre dynamique si besoin plus tard


In [3]:

# 📁 Préparation des dossiers de sortie
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# 🔄 Chargement des historiques si existants
hist_games_path = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)


In [4]:

# 📥 1. Téléchargement des matchs de la saison actuelle
print("\n📥 Téléchargement des nouveaux matchs pour la saison:", current_season)
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, current_season)
last_games_path = download_games_for_seasons([current_season], matchs_output_dir, run_timestamp, max_retries=10)


📥 Téléchargement des nouveaux matchs pour la saison: 2025-26
Extraction saison 2025-26


In [5]:

# 📊 2. Comparaison avec les données historiques pour trouver les nouveaux matchs
games_to_scrape = get_new_games(hist_games_path, last_games_path)
print(f"✅ {len(games_to_scrape)} nouveaux matchs trouvés à scraper.")

if games_to_scrape.empty:
    print("✅ Aucun nouveau match à scraper. Fin du script.")
    exit(0)

24 nouveaux matchs à traiter
✅ 24 nouveaux matchs trouvés à scraper.


In [6]:
games_to_scrape

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
218,22025,1610612748,MIA,Miami Heat,0022500178,2025-11-05,MIA @ DEN,L,241,112,...,5,33,38,26,13,4,14,20,-10.0,2025-26
219,22025,1610612738,BOS,Boston Celtics,0022500174,2025-11-05,BOS vs. WAS,W,241,136,...,12,28,40,31,12,3,5,24,29.0,2025-26
220,22025,1610612747,LAL,Los Angeles Lakers,0022500179,2025-11-05,LAL vs. SAS,W,242,118,...,12,26,38,23,8,5,12,30,2.0,2025-26
221,22025,1610612740,NOP,New Orleans Pelicans,0022500177,2025-11-05,NOP @ DAL,W,240,101,...,14,42,56,18,10,3,13,18,2.0,2025-26
222,22025,1610612745,HOU,Houston Rockets,0022500176,2025-11-05,HOU @ MEM,W,241,124,...,15,39,54,29,9,3,13,21,15.0,2025-26
223,22025,1610612739,CLE,Cleveland Cavaliers,0022500171,2025-11-05,CLE vs. PHI,W,241,132,...,10,24,34,33,9,7,14,22,11.0,2025-26
224,22025,1610612754,IND,Indiana Pacers,0022500173,2025-11-05,IND vs. BKN,L,239,103,...,16,40,56,29,6,5,16,23,-9.0,2025-26
225,22025,1610612765,DET,Detroit Pistons,0022500172,2025-11-05,DET vs. UTA,W,239,114,...,13,39,52,24,5,6,13,23,11.0,2025-26
226,22025,1610612762,UTA,Utah Jazz,0022500172,2025-11-05,UTA @ DET,L,240,103,...,11,33,44,24,7,2,15,17,-11.0,2025-26
227,22025,1610612744,GSW,Golden State Warriors,0022500181,2025-11-05,GSW @ SAC,L,241,116,...,10,34,44,28,10,6,18,26,-5.0,2025-26


In [7]:

# 💾 3. Merge historique + nouveaux matchs
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})
all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

games_output_path = save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'games_merged_all_seasons', run_timestamp)
print(f"✅ Jeux de matchs fusionnés sauvegardés dans {games_output_path}.")


✅ Jeux de matchs fusionnés sauvegardés dans data/01_bronze/games/games_merged_all_seasons_2025-11-07_23-30-01.csv.


In [8]:

# 🏀 4. Scraping des nouveaux boxscores
print(f"--- Traitement de la saison {current_season} ---")
season_df = games_to_scrape[games_to_scrape['SEASON'] == current_season]
season_output_dir = os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, current_season)
scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

--- Traitement de la saison 2025-26 ---
[DEBUG] Found 0 batch files for endpoint 'traditional' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'advanced' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'fourfactors' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'misc' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'scoring' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'usage' in season 2025-26
--------- 12 GAME_ID to scrap for season 2025-26 ---------
[1/12] GAME_ID: 0022500178 - 2025-11-05
[2/12] GAME_ID: 0022500174 - 2025-11-05
[3/12] GAME_ID: 0022500179 - 2025-11-05
[4/12] GAME_ID: 0022500177 - 2025-11-05
[5/12] GAME_ID: 0022500176 - 2025-11-05
[6/12] GAME_ID: 0022500171 - 2025-11-05
[7/12] GAME_ID: 0022500173 - 2025-11-05
[8/12] GAME_ID: 0022500172 - 2025-11-05
[9/12] GAME_ID: 0022500181 - 2025-11-05
[10/12] GAME_ID: 0022500180 - 2025-11-05
[11/12] GAME_ID: 0022500175 - 2025-11-05
[12/12] GAME_ID: 0022500182 - 2025-11-06


True

In [9]:
print("\n✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.")



✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.
